In [71]:
import requests 
import pandas as pd 
import numpy as np
import json
import time 
import os 
from pathlib import Path

In [72]:
os.makedirs("data/raw", exist_ok=True)
COUNTRIES = [
    "KEN", "TZA", "UGA", "ETH", "MOZ",       # East Africa
    "NGA", "GHA", "CIV", "SEN",               # West Africa
    "EGY", "MAR", "TUN",                      # North Africa
    "ZAF", "BWA", "ZMB",                      # Southern Africa
    "USA", "GBR", "DEU", "FRA", "JPN",        # Major advanced
    "CHN", "IND", "BRA", "MEX", "IDN",        # Large EM (IDN not TUR twice)
    "TUR", "THA", "MYS", "PHL", "VNM",        # Asia EM
    "SAU", "ARE", "NOR", "AUS", "CAN",        # Commodity exporters ← comma here
    "POL", "CZE", "HUN", "ROU",               # Eastern Europe
    "ARG", "CHL", "COL", "PER",               # Latin America
    "KOR", "SGP", "PAK", "BGD",              # Other Asia ← comma here
    "ISL", "NZL", "CRI",                      # Small open economies
]
INDICATORS= {
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",
    "FP.CPI.TOTL.ZG": "inflation",
    "SL.UEM.TOTL.ZS": "unemployment",
    "FR.INR.LEND": "lending_rate",
    "NE.TRD.GNFS.ZS": "trade_opennes",
    "BN.CAB.XOKA.GD.ZS": "govt_debt",
    "NY.GDP.PCAP.KD.ZG": "gdp_per_capita_growth"
}

START_YEAR= 1995
END_YEAR=2023
# RAW_PATH = Path(r"data/raw/wb_raw.csv")
# PROCESSED_PATH=Path(r"data/processed/panel.csv")
# TENSOR_PATH= Path(r"data/processed/tensor.npy")
# META_PATH = Path(r"data/processed/metadata.json")

In [73]:
def fetch_indicator(code, name, countries ,start_year,end_year):
    country_str=";".join(countries)
    url=(
        f"https://api.worldbank.org/v2/country/{country_str}/indicator/{code}?date={start_year}:{end_year}&format=json&per_page=10000"
    )
    resp=requests.get(url,timeout=30)
    resp.raise_for_status()
    payload=resp.json()
        
    if not isinstance(payload,list):
        print(f"Unexpected response for {code}: {payload}")
        return pd.DataFrame()
    if len(payload) < 2 or not payload[1]:
        print(f"Unexpected response for {code}: {payload}")
        print(f" Raw response: {payload[0]}")
        return pd.DataFrame()

    records=[
        {
            "country_iso3":item["countryiso3code"],
            "country_name": item["country"]["value"],
            "year": int(item["date"]),
            name: item["value"]
        }
        for item in payload[1]
    ]
    return pd.DataFrame(records)

panel=None

for code,name in INDICATORS.items():
    print(f"Fetching: {name}...")
    df= fetch_indicator(code,name,COUNTRIES,START_YEAR,END_YEAR)
    panel= df if panel is None else panel.merge(
        df[["country_iso3","year",name]],
        on=["country_iso3","year"],
        how="outer"

    )
    time.sleep(0.5)

panel = panel.sort_values(["country_iso3","year"]).reset_index(drop=True)

panel.to_csv("data/raw/wb_panel_raw.csv", index=False)

print(f"\nShape: {panel.shape}")
print(f"Countries: {panel['country_iso3'].nunique()}")
print(f"Years: {panel["year"].min()} - {panel['year'].max()}")

print("\n --- Missing Value Per Column-----")
for col in list(INDICATORS.values()):
    n= panel[col].isna().sum()
    pct= panel[col].isna().mean() * 100
    print(f" {col:<28} {n:>5} missing ({pct:.1f}%)")

print("\n--- Kenya rows(all years)--------")
print(panel[panel["country_iso3"]=="KEN"].to_string(index=False))
print("\nSaved")


Fetching: gdp_growth...
Fetching: inflation...
Fetching: unemployment...
Fetching: lending_rate...
Fetching: trade_opennes...
Fetching: govt_debt...
Fetching: gdp_per_capita_growth...

Shape: (1450, 10)
Countries: 50
Years: 1995 - 2023

 --- Missing Value Per Column-----
 gdp_growth                       0 missing (0.0%)
 inflation                       47 missing (3.2%)
 unemployment                     0 missing (0.0%)
 lending_rate                   422 missing (29.1%)
 trade_opennes                   51 missing (3.5%)
 govt_debt                       57 missing (3.9%)
 gdp_per_capita_growth            0 missing (0.0%)

--- Kenya rows(all years)--------
country_iso3 country_name  year  gdp_growth  inflation  unemployment  lending_rate  trade_opennes  govt_debt  gdp_per_capita_growth
         KEN        Kenya  1995    4.406217   1.554328         2.848     28.795833      71.745742 -17.446216               1.483737
         KEN        Kenya  1996    4.146839   8.864087         2.773   

In [74]:
missing_by_country=(
    panel.groupby("country_iso3")[["lending_rate","trade_opennes", "govt_debt", "inflation"]]
    .apply(lambda g: g.isna().mean() *100).round(1)
)
print(missing_by_country[missing_by_country["lending_rate"]> 0]. sort_values("lending_rate",ascending=False))

              lending_rate  trade_opennes  govt_debt  inflation
country_iso3                                                   
ARE                  100.0           20.7       96.6       44.8
FRA                  100.0            0.0        0.0        0.0
TUN                  100.0            0.0        0.0        0.0
DEU                  100.0            0.0        0.0        0.0
MAR                  100.0            0.0        0.0        0.0
GHA                  100.0            0.0        0.0        0.0
POL                  100.0            0.0        0.0        0.0
TUR                  100.0            0.0        0.0        0.0
SAU                  100.0            0.0        0.0        0.0
NOR                   62.1            0.0        0.0        0.0
SEN                   58.6            0.0        0.0        0.0
CIV                   58.6            0.0       34.5        0.0
ARG                   51.7            0.0        0.0       79.3
ETH                   51.7           55.

In [75]:
missing_by_country = (
    panel.groupby("country_iso3")[list(INDICATORS.values())]
    .apply(lambda g: g.isna().sum())
)

In [76]:
panel= panel.drop(columns=["lending_rate"])

In [77]:
panel.columns

Index(['country_iso3', 'country_name', 'year', 'gdp_growth', 'inflation',
       'unemployment', 'trade_opennes', 'govt_debt', 'gdp_per_capita_growth'],
      dtype='str')

In [78]:
panel.isna().sum()

country_iso3              0
country_name              0
year                      0
gdp_growth                0
inflation                47
unemployment              0
trade_opennes            51
govt_debt                57
gdp_per_capita_growth     0
dtype: int64

In [79]:
panel["country_name"].unique()

<StringArray>
['United Arab Emirates',            'Argentina',            'Australia',
           'Bangladesh',               'Brazil',             'Botswana',
               'Canada',                'Chile',                'China',
        'Cote d'Ivoire',             'Colombia',           'Costa Rica',
              'Czechia',              'Germany',     'Egypt, Arab Rep.',
             'Ethiopia',               'France',       'United Kingdom',
                'Ghana',              'Hungary',            'Indonesia',
                'India',              'Iceland',                'Japan',
                'Kenya',          'Korea, Rep.',              'Morocco',
               'Mexico',           'Mozambique',             'Malaysia',
              'Nigeria',               'Norway',          'New Zealand',
             'Pakistan',                 'Peru',          'Philippines',
               'Poland',              'Romania',         'Saudi Arabia',
              'Senegal',            '

In [80]:
panel.isna().groupby(panel["country_name"]).sum()

,country_iso3,country_name,year,gdp_growth,inflation,unemployment,trade_opennes,govt_debt,gdp_per_capita_growth
country_name,,,,,,,,,
Argentina,0,0,0,0,23,0,0,0,0
Australia,0,0,0,0,0,0,0,0,0
Bangladesh,0,0,0,0,0,0,0,0,0
Botswana,0,0,0,0,0,0,0,0,0
Brazil,0,0,0,0,0,0,0,0,0
Canada,0,0,0,0,0,0,0,0,0
Chile,0,0,0,0,0,0,0,0,0
China,0,0,0,0,0,0,0,0,0
Colombia,0,0,0,0,0,0,0,0,0


In [81]:
panel1=panel.copy()
panel1[panel1["country_name"]=="Rwanda"]

,country_iso3,country_name,year,gdp_growth,inflation,unemployment,trade_opennes,govt_debt,gdp_per_capita_growth


In [82]:
DROP = [
    "ARG",  # missing 2023 inflation + hyperinflation distortion
    "ARE",  # 28 govt debt, 13 inflation, 6 trade openness
    "NGA",  # 29 trade openness — structural reporting problem
    "ETH",  # 16 trade openness
    "CIV",  # 10 govt debt
    "RWA",  # only starts 2009 — too short
]
panel=panel[~panel["country_iso3"].isin(DROP)].reset_index(drop=True)

print(f"Shape: {panel.shape}")

Shape: (1305, 9)


In [84]:
panel

,country_iso3,country_name,year,gdp_growth,inflation,unemployment,trade_opennes,govt_debt,gdp_per_capita_growth
0,AUS,Australia,1995,3.890172,4.627764,8.473,37.646874,-5.063186,2.739531
1,AUS,Australia,1996,3.865675,2.615379,8.509,38.174021,-3.583603,2.612517
2,AUS,Australia,1997,3.917958,0.224895,8.367,37.921701,-2.729749,2.799586
3,AUS,Australia,1998,4.676814,0.860130,7.684,39.924066,-4.441118,3.638646
4,AUS,Australia,1999,5.028220,1.483139,6.876,38.960247,-5.583664,3.885499
...,...,...,...,...,...,...,...,...,...
1300,ZMB,Zambia,2019,1.441306,9.150316,5.538,68.791205,0.603691,-1.518950
1301,ZMB,Zambia,2020,-2.785055,15.733060,6.032,79.206849,11.790957,-5.567735
1302,ZMB,Zambia,2021,6.234922,22.020768,5.197,86.208511,11.900351,3.285755
1303,ZMB,Zambia,2022,5.211224,10.993204,5.993,69.297315,3.747847,2.343365


#### PRE-PROCESSING